In [ ]:
import pandas as pd
import numpy as np
import json
import networkx as nx
from sklearn.preprocessing import LabelEncoder
from pgmpy.models import DiscreteBayesianNetwork
from pgmpy.estimators import HillClimbSearch, BIC, BayesianEstimator # Keep BayesianEstimator import if needed elsewhere, but not used for fitting here
from pgmpy.inference import VariableElimination
from pgmpy.factors.discrete import TabularCPD
from sklearn.feature_selection import mutual_info_classif
from pgmpy.estimators import MaximumLikelihoodEstimator

# ------------------ Step 1: Load and Preprocess Data ------------------

df = pd.read_csv("Discretized_Dataset_with_Context.csv")
df = df.dropna()

# Manual ordinal encoding (including age)
ordinal_mapping = {'low': 0, 'medium': 1, 'high': 2}
ordinal_columns = [
    'Heart rate', 'Systolic Blood Pressure', 'Diastolic Blood Pressure',
    'Respiratory rate', 'Body temperature',
    'Pain severity - 0-10 verbal numeric rating [Score] - Reported', 'age'
]

for col in ordinal_columns:
    if col in df.columns:
        # Ensure mapping only applies to existing values
        df[col] = df[col].map(lambda x: ordinal_mapping.get(x, x)) # Handle potential non-mapped values gracefully if any

# Label encode other categorical variables
label_encoders = {}
for col in df.select_dtypes(include=['object']).columns:
    # Check if column still exists after potential drops/changes
    if col in df.columns:
        le = LabelEncoder()
        df[col] = le.fit_transform(df[col])
        label_encoders[col] = le

# ------------------ Step 2: Feature Selection ------------------

selected_columns = [
    'age', 'Heart rate', 'Systolic Blood Pressure', 'Diastolic Blood Pressure',
    'Respiratory rate', 'Body temperature',
    'Pain severity - 0-10 verbal numeric rating [Score] - Reported',
    'condition_context', 'triage_level',
    'has_stroke', 'has_sepsis_disorder', 'has_pneumonia',
    'has_dyspnea_finding', 'has_diabetes', 'has_myocardial_infarction',
    'has_chronic_congestive_heart_failure_disorder', 'has_fever_finding',
    'has_cardiac_arrest', 'has_history_of_cardiac_arrest_situation',
    'has_stress_finding',
    'has_coronary_heart_disease', 'has_pneumonia_disorder',
    'has_respiratory_distress_finding'
]

# Ensure only existing columns are selected
selected_columns = [col for col in selected_columns if col in df.columns]
df_subset = df[selected_columns].dropna()

# Convert all columns in df_subset to appropriate integer types after encoding/mapping
# This helps prevent type issues later
for col in df_subset.columns:
    # Check if column is not already numeric (int/float)
    if not pd.api.types.is_numeric_dtype(df_subset[col]):
        try:
            # Attempt conversion, handle potential errors if conversion fails
            df_subset[col] = pd.to_numeric(df_subset[col], errors='coerce').astype('Int64') # Use nullable Int64
        except Exception as e:
            print(f"Warning: Could not convert column {col} to numeric. Error: {e}")
    elif pd.api.types.is_float_dtype(df_subset[col]):
         # If it's float but should be int (like our encoded values)
         df_subset[col] = df_subset[col].astype(float).astype('Int64') # Convert float to nullable Int64
    elif pd.api.types.is_integer_dtype(df_subset[col]):
         # If already integer, ensure it's a standard type if needed, or use Int64
         df_subset[col] = df_subset[col].astype('Int64')


# Drop rows with NaN that might have been introduced by coercion/dropna
df_subset = df_subset.dropna()


# ------------------ Step 3: Structure Learning ------------------

hc = HillClimbSearch(df_subset)
scoring = BIC(df_subset)
# Ensure 'triage_level' exists before proceeding
if 'triage_level' not in df_subset.columns:
    raise ValueError("'triage_level' column not found in df_subset after preprocessing.")

learned_model = hc.estimate(scoring_method=scoring, max_indegree=15)

forced_parents = parents = ['Heart rate', 'Systolic Blood Pressure', 'has_myocardial_infarction', 'condition_context']



# Ensure forced parents exist in the dataframe before adding edges
valid_forced_parents = [p for p in forced_parents if p in df_subset.columns]

for parent in valid_forced_parents:
    if parent != 'triage_level' and (parent not in learned_model.predecessors('triage_level')):
        learned_model.add_edge(parent, 'triage_level')

# Cycle removal
while not nx.is_directed_acyclic_graph(learned_model):
    try:
        cycles = list(nx.simple_cycles(learned_model))
        if cycles:
            # Remove the first edge of the first found cycle
            edge_to_remove = list(zip(cycles[0], cycles[0][1:] + cycles[0][:1]))[0]
            print(f"Removing edge {edge_to_remove} to break cycle.")
            learned_model.remove_edge(*edge_to_remove)
        else:
            # Should not happen if is_directed_acyclic_graph is false, but break defensively
            print("Warning: No cycles found by nx.simple_cycles, but graph reported as cyclic.")
            break
    except Exception as e:
        print(f"Error during cycle removal: {e}")
        break


# ------------------ Step 4: Fit the Model ------------------

model = DiscreteBayesianNetwork(learned_model.edges())

# Initialize state_names_map dictionary
state_names_map = {}

# Define ordinal features
ordinal_features = [
    'Heart rate', 'Systolic Blood Pressure', 'Diastolic Blood Pressure',
    'Respiratory rate', 'Body temperature',
    'Pain severity - 0-10 verbal numeric rating [Score] - Reported', 'age'
]

# Build appropriate state_names map using actual unique values from df_subset
print("\n--- Defining State Names ---")
for col in model.nodes(): # Iterate over nodes actually in the learned model
    if col in df_subset.columns:
        unique_values = sorted(df_subset[col].unique())
        # Convert numpy types to standard Python types (int) for state names
        state_names_map[col] = [int(v) for v in unique_values]
        # print(f"State names for {col}: {state_names_map[col]}") # Optional: print states for each node
    else:
        print(f"Warning: Node {col} from learned structure not found in df_subset columns.")


# Print state names for key variables to verify
if 'age' in state_names_map:
    print("State names defined for age:", state_names_map['age'])
if 'condition_context' in state_names_map:
    print("State names defined for condition_context:", state_names_map['condition_context'])
if 'has_cardiac_arrest' in state_names_map:
    print("State names defined for has_cardiac_arrest:", state_names_map['has_cardiac_arrest'])
if 'has_myocardial_infarction' in state_names_map:
    print("State names defined for has_myocardial_infarction:", state_names_map['has_myocardial_infarction'])
if 'triage_level' in state_names_map:
    print("State names defined for triage_level:", state_names_map['triage_level'])


# Fit model using Maximum Likelihood Estimator
print("\n--- Fitting Model ---")
model.fit(
    df_subset,
    estimator=BayesianEstimator,
    prior_type="BDeu",
    equivalent_sample_size=50.0,  # 🔽 Reduce smoothing
    state_names=state_names_map
)

print("Model fitting complete.")

# Check CPDs
print("\n--- Checking CPDs ---")
model.check_model()
print("Model check passed.")
cpd = model.get_cpds('triage_level')
print("CPD shape:", cpd.values.shape)

# ------------------ Step 5: Bias the CPD (Commented Out) ------------------
# Bias section remains commented out as per original code

# ------------------ Step 6: Export Model ------------------
print("\n--- Exporting Model ---")
export_data = {
    "nodes": [str(node) for node in model.nodes()],
    "edges": [[str(u), str(v)] for u, v in model.edges()],
    "cpds": []
}

for cpd in model.get_cpds():
    # Ensure all state names are stringified for JSON compatibility
    state_map = {
        str(var): [str(state) for state in states]
        for var, states in cpd.state_names.items()
    }

    export_data["cpds"].append({
        "variable": str(cpd.variable),
        "evidence": [str(ev) for ev in cpd.variables[1:]], # cpd.evidence is equivalent
        "values": cpd.values.tolist(), # Convert numpy array to list
        "state_names": state_map
    })

try:
    with open("learned_bn_model.json", "w") as f:
        json.dump(export_data, f, indent=2)
    print("✅ Bayesian network trained and exported to learned_bn_model.json")
except Exception as e:
    print(f"Error exporting model: {e}")

# Print class distribution
if 'triage_level' in df_subset.columns:
    print("\nOriginal Triage Level Distribution in Training Data:")
    print(df_subset['triage_level'].value_counts(normalize=True))
else:
     print("\n'triage_level' not found for printing distribution.")


# ------------------ Step 7: Inference ------------------
print("\n--- Performing Inference ---")
infer = VariableElimination(model)

# Define Evidence Scenarios (Ensure values match learned states)
# Note: Keys not part of the model or not parents of triage_level will be ignored by prepare_evidence
cardiac_evidence = {
    "Heart rate": 1,
    "Systolic Blood Pressure": 1,
    "Diastolic Blood Pressure": 1, # Note: May not be a parent
    "Respiratory rate": 1,
    "Pain severity - 0-10 verbal numeric rating [Score] - Reported": 0,
    "has_myocardial_infarction": 1, # Is a parent
    "has_diabetes": 0,
    "condition_context": 0,
    "has_cardiac_arrest": 0,
    "age": 1
}

low_evidence = {
    "Heart rate": 1,
    "Systolic Blood Pressure": 1,
    "Diastolic Blood Pressure": 1, # Note: May not be a parent
    "Respiratory rate": 1,
    "Pain severity - 0-10 verbal numeric rating [Score] - Reported": 0,
    "condition_context": 0,
    "has_diabetes": 0,
    "has_cardiac_arrest": 0, # Changed from original
    "has_myocardial_infarction": 0, # Changed from original
    "age": 1
}

moderate_evidence = {
    "Heart rate": 1,
    "Systolic Blood Pressure": 1,
    "Diastolic Blood Pressure": 1, # Note: May not be a parent
    "Respiratory rate": 2,
    "Pain severity - 0-10 verbal numeric rating [Score] - Reported": 2,
    "condition_context": 2,
    "has_diabetes": 1,
    "has_cardiac_arrest": 1,
    "has_myocardial_infarction": 0, # Is a parent
    "age": 1
}

high_evidence = {
    "Heart rate": 2,
    "Systolic Blood Pressure": 0,
    "Diastolic Blood Pressure": 0, # Note: May not be a parent
    "Respiratory rate": 2,
    "Pain severity - 0-10 verbal numeric rating [Score] - Reported": 2,
    "condition_context": 0, # Assuming 'cardiac_related' maps to 0 based on typical encoding
    "has_diabetes": 1,
    "has_cardiac_arrest": 1,
    "has_myocardial_infarction": 0, # Is a parent
    "age": 2
}

# --- Get Parents and Baseline ---
try:
    parents = model.get_parents('triage_level')
    print("Parents of triage_level:", parents)
    triage_parents_set = set(parents)

    baseline_evidence = {}
    all_parents_found = True
    for p in parents:
        if p in df_subset.columns:
            mode_val = df_subset[p].mode()[0]
            if isinstance(mode_val, np.integer):
                mode_val = int(mode_val)
            elif not isinstance(mode_val, int): # Ensure it's int
                 mode_val = int(mode_val)
            baseline_evidence[p] = mode_val
        else:
            print(f"Error: Parent {p} not found in df_subset columns for baseline.")
            all_parents_found = False
            break # Stop if a parent is missing

    if not all_parents_found:
         raise ValueError("Cannot create baseline evidence, parent missing from data.")

    # print("Baseline Evidence:", baseline_evidence) # Optional print

except Exception as e:
    print(f"Error getting parents or baseline evidence: {e}")
    # Set defaults to allow code to run further, but inference will likely fail
    parents = []
    triage_parents_set = set()
    baseline_evidence = {}


# --- Define the prepare_evidence function ---
def prepare_evidence(evidence_dict, all_parents_set, base_evidence):
    """Ensures evidence has all parents, correct types, and no extra keys."""
    if not all_parents_set: # If parents couldn't be determined
        print("Warning: Cannot prepare evidence, parent set is empty.")
        return None

    prepared_evidence = {}
    # Start with base evidence for all parents
    for p in all_parents_set:
         if p in base_evidence:
             prepared_evidence[p] = base_evidence[p] # Already correct type
         else:
             print(f"Warning: Parent {p} missing from baseline_evidence during preparation.")
             # Decide how to handle: skip, error, or use a default? Using 0 for now.
             prepared_evidence[p] = 0


    # Override with values from the specific evidence dict if parent exists
    for key, value in evidence_dict.items():
        if key in all_parents_set:
            # Ensure correct type (standard Python int)
            try:
                if isinstance(value, np.integer):
                    prepared_evidence[key] = int(value)
                elif isinstance(value, (int, float)):
                     prepared_evidence[key] = int(value) # Cast to int
                else:
                     # Attempt conversion if it's string-like number? Risky.
                     prepared_evidence[key] = int(value)
            except (ValueError, TypeError):
                 print(f"Warning: Could not convert value '{value}' for key '{key}' to int. Using baseline value.")
                 prepared_evidence[key] = base_evidence.get(key, 0) # Fallback to baseline or 0

    # Final check: ensure all parents are still in the dict
    for p in all_parents_set:
        if p not in prepared_evidence:
             print(f"Error: Parent {p} was lost during evidence preparation.")
             return None # Indicate error

    return prepared_evidence

# --- Prepare the specific evidence dictionaries ---
low_evidence_prepared = prepare_evidence(low_evidence, triage_parents_set, baseline_evidence)
moderate_evidence_prepared = prepare_evidence(moderate_evidence, triage_parents_set, baseline_evidence)
high_evidence_prepared = prepare_evidence(high_evidence, triage_parents_set, baseline_evidence)
cardiac_evidence_prepared = prepare_evidence(cardiac_evidence, triage_parents_set, baseline_evidence)


# --- Perform Inference with Prepared Evidence ---
print("\n--- Inference with Prepared Evidence ---")

print("Low Evidence (Prepared):", low_evidence_prepared)
if low_evidence_prepared:
    try:
        result1 = infer.query(variables=["triage_level"], evidence=low_evidence_prepared, show_progress=False)
        print("Result (Low Evidence):")
        print(result1)
    except Exception as e:
        print(f"Error during low evidence query: {e}")
else:
    print("Skipping low evidence query due to preparation error.")


print("Moderate Evidence (Prepared):", moderate_evidence_prepared)
if moderate_evidence_prepared:
    try:
        result2 = infer.query(variables=["triage_level"], evidence=moderate_evidence_prepared, show_progress=False)
        print("Result (Moderate Evidence):")
        print(result2)
    except Exception as e:
        print(f"Error during moderate evidence query: {e}")
else:
    print("Skipping moderate evidence query due to preparation error.")


print("High Evidence (Prepared):", high_evidence_prepared)
if high_evidence_prepared:
    try:
        result3 = infer.query(variables=["triage_level"], evidence=high_evidence_prepared, show_progress=False)
        print("Result (High Evidence):")
        print(result3)
    except Exception as e:
        print(f"Error during high evidence query: {e}")
else:
    print("Skipping high evidence query due to preparation error.")

print("Cardiac Evidence (Prepared):", cardiac_evidence_prepared)
if cardiac_evidence_prepared:
    try:
        result_cardiac = infer.query(variables=["triage_level"], evidence=cardiac_evidence_prepared, show_progress=False)
        print("Result (Cardiac Evidence):")
        print(result_cardiac)
    except Exception as e:
        print(f"Error during cardiac evidence query: {e}")
else:
    print("Skipping cardiac evidence query due to preparation error.")


# ------------------ Step 8: Analysis ------------------
print("\n--- Further Analysis ---")

# Print impact analysis loop (check if triage_level CPD exists)
triage_cpd = model.get_cpds('triage_level')
if triage_cpd:
    print(f"\nParents of triage_level used for impact check: {triage_cpd.variables[1:]}")
    for var in ["has_myocardial_infarction", "has_cardiac_arrest", "condition_context"]:
        if var in triage_cpd.variables:
            # idx = triage_cpd.variables.index(var) - 1 # Index relative to evidence list
            print(f"\n>>> Analyzing Impact of {var} on triage_level:")
            print(f"   Variable Cardinality: {triage_cpd.cardinality[triage_cpd.variables.index(var)]}")
            print(f"   Evidence Variables: {triage_cpd.variables[1:]}")
            print(f"   Evidence Cardinalities: {triage_cpd.cardinality[1:]}")
            print(f"   CPD Shape (triage_level_card, *evidence_card): {triage_cpd.values.shape}")
            # Simplified check: Does changing this var (0 vs 1) while others are baseline change outcome?
            if var in baseline_evidence: # Ensure var is a parent
                 try:
                     evidence_var_0 = baseline_evidence.copy()
                     evidence_var_0[var] = 0 # Assuming 0 is a valid state
                     evidence_var_0 = prepare_evidence(evidence_var_0, triage_parents_set, baseline_evidence) # Re-prepare
                     if evidence_var_0 and 0 in state_names_map.get(var,[]):
                         res_0 = infer.query(variables=["triage_level"], evidence=evidence_var_0, show_progress=False)
                         print(f"   Inference with {var}=0: {res_0.values.tolist()}")

                     evidence_var_1 = baseline_evidence.copy()
                     evidence_var_1[var] = 1 # Assuming 1 is a valid state
                     evidence_var_1 = prepare_evidence(evidence_var_1, triage_parents_set, baseline_evidence) # Re-prepare
                     if evidence_var_1 and 1 in state_names_map.get(var,[]):
                         res_1 = infer.query(variables=["triage_level"], evidence=evidence_var_1, show_progress=False)
                         print(f"   Inference with {var}=1: {res_1.values.tolist()}")
                 except Exception as e:
                     print(f"   Error during simplified impact inference for {var}: {e}")

            # Don't break, show analysis for all specified vars if they are parents
        else:
             print(f"\n>>> {var} is not a direct parent of triage_level in the learned model.")

else:
    print("\nCPD for 'triage_level' not found.")


# Mutual Information Scores
print("\n--- Mutual Information Scores ---")
if 'triage_level' in df_subset.columns:
    X = df_subset.drop(columns=["triage_level"])
    y = df_subset["triage_level"]

    # Ensure all features in X are discrete (or treat them as such)
    # Convert X to int type just to be safe for MI calculation
    X = X.astype(int)
    discrete_features_mask = [True] * X.shape[1] # Treat all as discrete

    try:
        mi_scores = mutual_info_classif(X, y, discrete_features=discrete_features_mask)
        mi_df = pd.DataFrame({"Feature": X.columns, "Mutual Information": mi_scores})
        mi_df = mi_df.sort_values(by="Mutual Information", ascending=False)
        print("Top features by mutual information with triage_level:")
        print(mi_df.head(20))
    except Exception as e:
        print(f"Error calculating Mutual Information: {e}")
else:
    print("'triage_level' not found for MI calculation.")


# Print CPD for triage_level again
print("\n--- CPD for triage_level ---")
if triage_cpd:
    print(triage_cpd)
else:
    print("CPD for 'triage_level' not found.")

# Crosstab
print("\n--- Crosstab Analysis ---")
if 'has_cardiac_arrest' in df_subset.columns and 'triage_level' in df_subset.columns:
    try:
        ct = pd.crosstab(df_subset['has_cardiac_arrest'], df_subset['triage_level'], normalize='index')
        print("Triage level distribution by cardiac arrest flag:")
        print(ct)
    except Exception as e:
        print(f"Error creating crosstab for has_cardiac_arrest: {e}")
else:
    print("Required columns for cardiac arrest crosstab not found.")

if 'has_myocardial_infarction' in df_subset.columns and 'triage_level' in df_subset.columns:
    try:
        ct_mi = pd.crosstab(df_subset['has_myocardial_infarction'], df_subset['triage_level'], normalize='index')
        print("\nTriage level distribution by myocardial infarction flag:")
        print(ct_mi)
    except Exception as e:
        print(f"Error creating crosstab for has_myocardial_infarction: {e}")
else:
    print("Required columns for myocardial infarction crosstab not found.")

print("\n--- End of Script ---")

print(df_subset[df_subset["has_myocardial_infarction"] == 1]["triage_level"].value_counts(normalize=True))

# Check full evidence combo count for cardiac_evidence
cardiac_mask = (
    (df_subset["Heart rate"] == 1) &
    (df_subset["condition_context"] == 0) &
    (df_subset["Systolic Blood Pressure"] == 1) &
    (df_subset["has_myocardial_infarction"] == 1)
)
print("Rows matching cardiac evidence:", cardiac_mask.sum())

INFO:pgmpy: Datatype (N=numerical, C=Categorical Unordered, O=Categorical Ordered) inferred from data: 
 {'age': 'N', 'Heart rate': 'N', 'Systolic Blood Pressure': 'N', 'Diastolic Blood Pressure': 'N', 'Respiratory rate': 'N', 'Body temperature': 'N', 'Pain severity - 0-10 verbal numeric rating [Score] - Reported': 'N', 'condition_context': 'N', 'triage_level': 'N', 'has_stroke': 'N', 'has_sepsis_disorder': 'N', 'has_pneumonia': 'N', 'has_dyspnea_finding': 'N', 'has_diabetes': 'N', 'has_myocardial_infarction': 'N', 'has_chronic_congestive_heart_failure_disorder': 'N', 'has_fever_finding': 'N', 'has_cardiac_arrest': 'N', 'has_history_of_cardiac_arrest_situation': 'N', 'has_stress_finding': 'N', 'has_coronary_heart_disease': 'N', 'has_pneumonia_disorder': 'N', 'has_respiratory_distress_finding': 'N'}
INFO:pgmpy: Datatype (N=numerical, C=Categorical Unordered, O=Categorical Ordered) inferred from data: 
 {'age': 'N', 'Heart rate': 'N', 'Systolic Blood Pressure': 'N', 'Diastolic Blood Pres

  0%|          | 0/1000000 [00:00<?, ?it/s]

INFO:pgmpy: Datatype (N=numerical, C=Categorical Unordered, O=Categorical Ordered) inferred from data: 
 {'age': 'N', 'Heart rate': 'N', 'Systolic Blood Pressure': 'N', 'Diastolic Blood Pressure': 'N', 'Respiratory rate': 'N', 'Body temperature': 'N', 'Pain severity - 0-10 verbal numeric rating [Score] - Reported': 'N', 'condition_context': 'N', 'triage_level': 'N', 'has_stroke': 'N', 'has_sepsis_disorder': 'N', 'has_pneumonia': 'N', 'has_dyspnea_finding': 'N', 'has_diabetes': 'N', 'has_myocardial_infarction': 'N', 'has_chronic_congestive_heart_failure_disorder': 'N', 'has_fever_finding': 'N', 'has_cardiac_arrest': 'N', 'has_history_of_cardiac_arrest_situation': 'N', 'has_stress_finding': 'N', 'has_coronary_heart_disease': 'N', 'has_pneumonia_disorder': 'N', 'has_respiratory_distress_finding': 'N'}


Removing edge ('Systolic Blood Pressure', 'triage_level') to break cycle.
Removing edge ('has_myocardial_infarction', 'triage_level') to break cycle.

--- Defining State Names ---
State names defined for age: [0, 1, 2]
State names defined for condition_context: [0, 1, 2, 3, 4, 5]
State names defined for has_cardiac_arrest: [0, 1]
State names defined for has_myocardial_infarction: [0, 1]
State names defined for triage_level: [0, 1, 2]

--- Fitting Model ---
Model fitting complete.

--- Checking CPDs ---
Model check passed.
CPD shape: (3, 3, 6)

--- Exporting Model ---
✅ Bayesian network trained and exported to learned_bn_model.json

Original Triage Level Distribution in Training Data:
triage_level
0    0.503454
1     0.33247
2    0.164076
Name: proportion, dtype: Float64

--- Performing Inference ---
Parents of triage_level: ['Heart rate', 'condition_context']

--- Inference with Prepared Evidence ---
Low Evidence (Prepared): {'condition_context': 0, 'Heart rate': 1}
Result (Low Evidenc